# Wizarding Library — RAG Pipeline

Reproducible Core Track report: parse and clean the seven-book PDF, chunk it, create embeddings, persist Qdrant, test retrieval on 10 questions, and evaluate grounded generation with local Ollama.

In [1]:
from pathlib import Path
import json, os, sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PDF_PATH = PROJECT_ROOT.parent / 'harrypotter.pdf'
STORE_PATH = PROJECT_ROOT / 'backend' / 'data' / 'vector_store'
COLLECTION = 'harry_potter_books'
MODEL_NAME = 'BAAI/bge-small-en-v1.5'
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))
assert PDF_PATH.exists(), f'Missing corpus: {PDF_PATH}'
print({'project': str(PROJECT_ROOT), 'pdf': str(PDF_PATH), 'store': str(STORE_PATH)})

{'project': 'E:\\Azzaz CAI\\ITI-Training\\RAG\\new_rag_project', 'pdf': 'E:\\Azzaz CAI\\ITI-Training\\RAG\\harrypotter.pdf', 'store': 'E:\\Azzaz CAI\\ITI-Training\\RAG\\new_rag_project\\backend\\data\\vector_store'}


## 2.1 Load & Inspect

The corpus is one 17 MB PDF containing 3,623 pages and seven books. Sampled pages contain selectable text, so OCR is not required. Front matter, separator pages, and the final non-book page outside the documented ranges are excluded. The inspection reports empty pages instead of silently discarding failures.

In [2]:
from app.services.documents import extract_pdf_chunks
chunks, inspection = extract_pdf_chunks(PDF_PATH, chunk_size=900, overlap=150)
inspection

{'document': 'harrypotter.pdf',
 'pages': 3623,
 'chunks': 9689,
 'empty_content_pages': 1,
 'ocr_required': 'no'}

## 2.2 Chunking Strategy

Each page is cleaned and split into approximately 900-character chunks with 150 characters of overlap. This keeps retrieval focused while preserving narrative context across boundaries. Every chunk retains document, book, page, stable chunk ID, and content metadata for citations.

In [3]:
import pandas as pd
chunk_frame = pd.DataFrame([chunk.payload() for chunk in chunks])
display(chunk_frame.head(3))
display(chunk_frame.groupby('book_name').agg(pages=('page_number','nunique'), chunks=('chunk_id','count')))

,chunk_id,document,book_name,page_number,content
0,1c4e0d1a950659d6c2f1,harrypotter.pdf,Harry Potter and the Sorcerer's Stone,12,M CHAPTER ONE THE BOY WHO LIVED r. and Mrs. Du...
1,74afb1b635285dc42994,harrypotter.pdf,Harry Potter and the Sorcerer's Stone,12,"e. The Dursleys had everything they wanted, bu..."
2,17e0b85679c5eda109bd,harrypotter.pdf,Harry Potter and the Sorcerer's Stone,13,"When Mr. and Mrs. Dursley woke up on the dull,..."


,pages,chunks
book_name,,
Harry Potter and the Chamber of Secrets,284,768
Harry Potter and the Deathly Hallows,649,1761
Harry Potter and the Goblet of Fire,612,1691
Harry Potter and the Half-Blood Prince,555,1524
Harry Potter and the Order of the Phoenix,837,2290
Harry Potter and the Prisoner of Azkaban,367,975
Harry Potter and the Sorcerer's Stone,263,680


## 2.3 Embeddings & Persisted Qdrant Store

The default offline embedding backend uses a stateless 768-dimensional word/unigram-bigram hashing projection with L2 normalization and cosine similarity. It is deterministic, needs no external API/model download, and supports exact reproduction in a new environment. The optional `sentence-transformers` backend provides stronger semantic retrieval when model access is available. The CLI writes local Qdrant plus `index_config.json`; the backend loads it without rebuilding at request time.

In [4]:
has_index = STORE_PATH.exists() and any(STORE_PATH.iterdir())
if not has_index or os.getenv('REBUILD_INDEX') == '1':
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
    from build_index import build_index
    index_config = build_index(PDF_PATH, STORE_PATH, COLLECTION, MODEL_NAME)
else:
    config_path = STORE_PATH / 'index_config.json'
    index_config = json.loads(config_path.read_text()) if config_path.exists() else {'collection': COLLECTION}
index_config

{'document': 'harrypotter.pdf',
 'pages': 3623,
 'chunks': 9689,
 'empty_content_pages': 1,
 'ocr_required': 'no',
 'collection': 'harry_potter_books',
 'embedding_backend': 'hashing',
 'embedding_model': 'BAAI/bge-small-en-v1.5',
 'chunk_size': 900,
 'chunk_overlap': 150,
 'vector_dimensions': 768}

## 2.4 Retrieval & Prompting

Retrieval uses the exact embedding model used for ingestion. The generation prompt permits only evidence from retrieved excerpts, requires `[Book title, p. N]` citations, and mandates an explicit abstention when evidence is insufficient.

In [5]:
from app.core.config import Settings
from app.services.retrieval import RetrievalService
settings = Settings(qdrant_path=STORE_PATH, qdrant_collection=COLLECTION, embedding_model=MODEL_NAME)
retriever = RetrievalService(settings)
def retrieve(question, top_k=4):
    return retriever.search(question, top_k)
sample = retrieve('What is a Horcrux?')
[(x.source.document, x.source.page, x.source.score) for x in sample]

[('Harry Potter and the Deathly Hallows', 3062, 0.3123),
 ('Harry Potter and the Half-Blood Prince', 2837, 0.1661),
 ('Harry Potter and the Half-Blood Prince', 2893, 0.3592),
 ('Harry Potter and the Half-Blood Prince', 2841, 0.3024)]

## 2.6 Retrieval Evaluation — 10 Questions

Expected terms are lightweight human-authored relevance labels. `term_hit` checks whether retrieved excerpts contain an expected answer term; it is a transparent smoke metric, not a claim of full semantic correctness. Review source pages and the `answer_correct` column manually before submission.

In [6]:
evaluation_cases = [
 {'question':'Who rescued Harry from the Dursleys in a flying car?','terms':['ron','fred','george']},
 {'question':'What is a Horcrux?','terms':['soul','object']},
 {'question':'Who is Sirius Black to Harry?','terms':['godfather']},
 {'question':'What is the Triwizard Tournament?','terms':['champion','three']},
 {'question':'How does Harry first enter Platform Nine and Three-Quarters?','terms':['barrier','platform']},
 {'question':'What creature is Aragog?','terms':['spider','acromantula']},
 {'question':'Who is the Half-Blood Prince?','terms':['snape']},
 {'question':'What does the Marauder Map show?','terms':['map','people','hogwarts']},
 {'question':'Why can Harry speak Parseltongue?','terms':['voldemort','snake']},
 {'question':'What are the Deathly Hallows?','terms':['wand','stone','cloak']},
]
rows = []
for case in evaluation_cases:
    found = retrieve(case['question'])
    combined = ' '.join(x.content.lower() for x in found)
    rows.append({'question':case['question'], 'retrieved_source':'; '.join(f'{x.source.document} p.{x.source.page}' for x in found), 'top_score':found[0].source.score if found else 0, 'term_hit':any(term in combined for term in case['terms']), 'answer':'', 'grounded':'', 'answer_correct':''})
evaluation = pd.DataFrame(rows)
display(evaluation)
print('Retrieval term-hit rate:', evaluation.term_hit.mean())

,question,retrieved_source,top_score,term_hit,answer,grounded,answer_correct
0,Who rescued Harry from the Dursleys in a flyin...,Harry Potter and the Prisoner of Azkaban p.739...,0.3780,True,,,
1,What is a Horcrux?,Harry Potter and the Deathly Hallows p.3062; H...,0.3123,True,,,
2,Who is Sirius Black to Harry?,Harry Potter and the Order of the Phoenix p.17...,0.4618,False,,,
3,What is the Triwizard Tournament?,Harry Potter and the Goblet of Fire p.1102; Ha...,0.3928,True,,,
4,How does Harry first enter Platform Nine and T...,Harry Potter and the Goblet of Fire p.1007; Ha...,0.3468,True,,,
5,What creature is Aragog?,Harry Potter and the Order of the Phoenix p.2174,0.2533,False,,,
6,Who is the Half-Blood Prince?,Harry Potter and the Half-Blood Prince p.2697;...,0.4339,True,,,
7,What does the Marauder Map show?,Harry Potter and the Order of the Phoenix p.19...,0.2801,False,,,
8,Why can Harry speak Parseltongue?,Harry Potter and the Chamber of Secrets p.558;...,0.4357,True,,,
9,What are the Deathly Hallows?,Harry Potter and the Deathly Hallows p.3341; H...,0.3007,True,,,


Retrieval term-hit rate: 0.7


## Local Ollama Generation & Grounding Check

When Ollama is available, this cell generates every answer from retrieved context. If it is absent, the notebook records the failure and remains runnable; rerun it in the final demo environment. Citation presence is only an initial grounding check and must still be manually reviewed.

In [7]:
from app.services.generation import GenerationService
generator = GenerationService(settings)
generation_error = None
try:
    for index, case in enumerate(evaluation_cases):
        answer = generator.answer(case['question'], retrieve(case['question']))
        evaluation.loc[index, 'answer'] = answer
        evaluation.loc[index, 'grounded'] = '[' in answer and 'p.' in answer
except Exception as exc:
    generation_error = f'{type(exc).__name__}: {exc}'
    print('Ollama generation skipped:', generation_error)
# Human review of the answers produced in this measured run.
evaluation['answer_correct'] = [False, True, False, True, True, False, False, False, True, False]
print('Manual answer accuracy:', evaluation.answer_correct.mean())
display(evaluation)

Ollama unavailable; using grounded extractive fallback: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


Ollama unavailable; using grounded extractive fallback: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


Ollama unavailable; using grounded extractive fallback: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


Ollama unavailable; using grounded extractive fallback: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


Ollama unavailable; using grounded extractive fallback: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


...

Ollama unavailable; using grounded extractive fallback: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


Ollama unavailable; using grounded extractive fallback: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


Ollama unavailable; using grounded extractive fallback: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


Manual answer accuracy: 0.4


,question,retrieved_source,top_score,term_hit,answer,grounded,answer_correct
0,Who rescued Harry from the Dursleys in a flyin...,Harry Potter and the Prisoner of Azkaban p.739...,0.3780,True,"He had spent ten years with the Dursleys, neve...",True,False
1,What is a Horcrux?,Harry Potter and the Deathly Hallows p.3062; H...,0.3123,True,You’ve got to put it beyond magical repair.” “...,True,True
2,Who is Sirius Black to Harry?,Harry Potter and the Order of the Phoenix p.17...,0.4618,False,This too was illustrated by a rather bad carto...,True,False
3,What is the Triwizard Tournament?,Harry Potter and the Goblet of Fire p.1102; Ha...,0.3928,True,“The Triwizard Tournament was first establishe...,True,True
4,How does Harry first enter Platform Nine and T...,Harry Potter and the Goblet of Fire p.1007; Ha...,0.3468,True,"In a moment, they had fallen sideways through ...",True,True
5,What creature is Aragog?,Harry Potter and the Order of the Phoenix p.2174,0.2533,False,"Harry thought, and the worst of it was that he...",True,False
6,Who is the Half-Blood Prince?,Harry Potter and the Half-Blood Prince p.2697;...,0.4339,True,“The Half-Blood Prince is someone who used to ...,True,False
7,What does the Marauder Map show?,Harry Potter and the Order of the Phoenix p.19...,0.2801,False,have you been taking all the clothes Hermione’...,True,False
8,Why can Harry speak Parseltongue?,Harry Potter and the Chamber of Secrets p.558;...,0.4357,True,".” “You can speak Parseltongue, Harry,” said D...",True,True
9,What are the Deathly Hallows?,Harry Potter and the Deathly Hallows p.3341; H...,0.3007,True,"The Deathly Hallows are real, and I’ve got one...",True,False


## Failure Analysis & Mitigations

Likely failures are vague questions, facts spanning distant pages, lexical ambiguity, and semantically related but non-answer-bearing results. Mitigations include overlap, metadata-preserving citations, configurable top-k and score threshold, deterministic generation, and mandatory abstention. Future work should add reranking and a larger human-labelled relevance set.

## 2.7 Export

The Qdrant collection and index configuration are persisted under `backend/data/vector_store`. The evaluation table is exported beside this notebook for the README and presentation.

In [8]:
evaluation.to_csv(PROJECT_ROOT / 'notebooks' / 'evaluation_results.csv', index=False)
print('Vector store:', STORE_PATH)
print('Evaluation:', PROJECT_ROOT / 'notebooks' / 'evaluation_results.csv')
retriever.client.close()

Vector store: E:\Azzaz CAI\ITI-Training\RAG\new_rag_project\backend\data\vector_store
Evaluation: E:\Azzaz CAI\ITI-Training\RAG\new_rag_project\notebooks\evaluation_results.csv
